In [1]:
import orekit_jpype as orekit
import jpype.imports 

orekit.initVM(jvmpath="C://Program Files//Eclipse Adoptium//jdk-17.0.18.8-hotspot//bin//server//jvm.dll",
              additional_classpaths=["D://GitHub//contigo_edr//contigo//orekit_utils//java//orekit-utils-1.0.0.jar"])


In [2]:
import pandas as pd
import zipfile
from orekit_jpype.pyhelpers import setup_orekit_data, download_orekit_data_curdir
download_orekit_data_curdir()
setup_orekit_data(from_pip_library=False)

In [3]:
from contigo_orekit.orekitutils import EphemerisBatchHelper

In [4]:
from org.orekit.time import AbsoluteDate, TimeScalesFactory
import numpy as np

def get_sun_moon_ecef_java(times):

    utc = TimeScalesFactory.getGPS()

    first_dt = times[0]
    ref_date = AbsoluteDate(
        first_dt.year,
        first_dt.month,
        first_dt.day,
        first_dt.hour,
        first_dt.minute,
        float(first_dt.second),
        utc
    )

    offsets = np.array(
        [(t - first_dt).total_seconds() for t in times],
        dtype=np.float64
    )

    # Single JVM call (entire batch handled in Java)
    result = EphemerisBatchHelper.getSunMoonECEF(ref_date, offsets)

    # Convert to NumPy once
    return np.array(result, dtype=np.float64)

In [9]:
from contigo.forces.third_body_acc import ThirdBody
from contigo.constellation import Constellation
from contigo.forces.tba_utils import tba_pairwise_numba

In [11]:
from org.orekit.frames import FramesFactory
from org.orekit.time import AbsoluteDate, TimeScalesFactory
from org.orekit.utils import IERSConventions
from org.orekit.bodies import CelestialBodyFactory

import numpy as np


def get_sun_moon_ecef(times):
    """
    Parameters
    ----------
    times : iterable of datetime-like objects
        Must contain year, month, day, hour, minute, second attributes
        (e.g., pandas Timestamp or python datetime).

    Returns
    -------
    np.ndarray
        Array of shape (2, N, 3)
        [0, i, :] → Sun position in ECEF (meters)
        [1, i, :] → Moon position in ECEF (meters)
    """

    N = len(times)
    output = np.empty((2, N, 3), dtype=np.float64)

    # ------------------------------------------------------------------
    # Frames and bodies (cached once)
    # ------------------------------------------------------------------
    eci = FramesFactory.getEME2000()
    ecef = FramesFactory.getITRF(IERSConventions.IERS_2010, True)

    sun = CelestialBodyFactory.getSun()
    moon = CelestialBodyFactory.getMoon()

    # ------------------------------------------------------------------
    # Time handling (fast construction using shiftedBy)
    # ------------------------------------------------------------------
    #utc = TimeScalesFactory.getUTC()
    utc = TimeScalesFactory.getGPS()

    first_dt = times[0]
    ref_date = AbsoluteDate(
        first_dt.year,
        first_dt.month,
        first_dt.day,
        first_dt.hour,
        first_dt.minute,
        float(first_dt.second),
        utc
    )

    # Compute offsets in seconds from first epoch
    offsets = [(t - first_dt).total_seconds() for t in times]

    # ------------------------------------------------------------------
    # Main loop (minimal JVM crossings)
    # ------------------------------------------------------------------
    for i, dt_sec in enumerate(offsets):

        date = ref_date.shiftedBy(float(dt_sec))

        # Get PV in ECI once
        sun_pv = sun.getPVCoordinates(date, eci)
        moon_pv = moon.getPVCoordinates(date, eci)

        #return sun_pv

        # Static transform (position only → faster)
        transform = eci.getStaticTransformTo(ecef, date)

        sun_ecef = transform.transformPosition(sun_pv.getPosition())
        moon_ecef = transform.transformPosition(moon_pv.getPosition())

        #sp = sun_ecef.getPosition()
        #mp = moon_ecef.getPosition()

        output[0, i, 0] = sun_ecef.getX()
        output[0, i, 1] = sun_ecef.getY()
        output[0, i, 2] = sun_ecef.getZ()

        output[1, i, 0] = moon_ecef.getX()
        output[1, i, 1] = moon_ecef.getY()
        output[1, i, 2] = moon_ecef.getZ()

    return output

In [7]:
# read in data
sw_d = pd.read_hdf('./data/ESA_pod.hdf')
ore_d = pd.read_hdf('./data/ore_d.hdf')
ore_a = ore_d[[ 'ecef_sn_ax', 'ecef_sn_ay', 'ecef_sn_az', 
               'ecef_mn_ax', 'ecef_mn_ay', 'ecef_mn_az']].to_numpy()
hdf_sc = Constellation(state_file=r'./data/ESA_pod.hdf',
                    time_col='DateTime', x_col='x', y_col='y', z_col='z',
                    vx_col='vx', vy_col='vy', vz_col='vz',
                    sc_id_col='filename', sc_fn_slc=slice(-11,-8),
                    tscale_input='GPS',
                    sc_mass=4.3e+02, cr=1.8, srp_area=1)

In [17]:
%%timeit -n 1 -r 1
# create a Constellation object from the ESA POD file
# and calculate thirdbody acceleration from ThirdBody
# Chain Initialization and acceleration derivation together
acc = ThirdBody(body=['SUN','MOON']).acceleration(hdf_sc)

1.31 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [14]:
%%timeit -n 1 -r 1
t = pd.to_datetime(sw_d['DateTime'].to_numpy())
ss_pos = get_sun_moon_ecef(t)/1000.
sc_pos = sw_d[['x','y','z']].to_numpy()


ore_acc = tba_pairwise_numba(sc_pos, ss_pos, [132712440041.279419,4902.800118])

9.15 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [10]:
%%timeit -n 1 -r 1
t = pd.to_datetime(sw_d['DateTime'].to_numpy())
ss_pos = get_sun_moon_ecef_java(t)/1000.
sc_pos = sw_d[['x','y','z']].to_numpy()


ore_acc = tba_pairwise_numba(sc_pos, ss_pos, [132712440041.279419,4902.800118])

1.36 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


CONTIGO - First run 2.89 s
CONTIGO - Second run 1.32
OREKIT - First run 7.88 s
OREKIT - Second run 2.18 s 

In [11]:
t = pd.to_datetime(sw_d['DateTime'].to_numpy())
ss_pos = get_sun_moon_ecef_java(t)/1000.
sc_pos = sw_d[['x','y','z']].to_numpy()


ore_acc = tba_pairwise_numba(sc_pos, ss_pos, [132712440041.279419,4902.800118])
acc = ThirdBody(body=['SUN','MOON']).acceleration(hdf_sc)

In [12]:
print(np.allclose(ore_acc, acc['ESA']))
print(np.array_equal(ore_acc, acc['ESA']))

print(ore_acc[0,0,:])
print(acc['ESA'][0,0,:])
print(ore_a[0,0:3]/1000.)


True
False
[-9.00593698e-11  2.80000067e-10  9.67685633e-12]
[-9.00593868e-11  2.80000065e-10  9.67684671e-12]
[-9.00593697e-11  2.80000067e-10  9.67685636e-12]


In [ ]:
import pandas as pd
sw_d = pd.read_hdf('./data/ESA_pod.hdf')
t = pd.to_datetime(sw_d['DateTime'].to_numpy())
get_sun_moon_ecef_java(t)